# Setup & Dependencies

In [ ]:
#%matplotlib inline

In [ ]:
! pip install -q transformers[sentencepiece] fastbook fastai nbdev plum-dispatch evaluate seqeval onnxruntime onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 127.0 MB/s eta 0:00:00


In [ ]:
!pip install -q onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/msi1427/blurr.git
%cd blurr

Cloning into 'blurr'...
remote: Enumerating objects: 5063, done.
remote: Counting objects: 100% (890/890), done.
remote: Compressing objects: 100% (310/310), done.
remote: Total 5063 (delta 701), reused 699 (delta 573), pack-reused 4173 (from 1)
Receiving objects: 100% (5063/5063), 27.00 MiB | 21.58 MiB/s, done.
Resolving deltas: 100% (3932/3932), done.
/content/blurr


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig
from fastai.text.all import *
from blurr.text.data.all import *
from blurr.text.modeling.all import *

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd "/content/drive/MyDrive/Startups_Recognize"

/content/drive/MyDrive/Startups_Recognize


In [ ]:
#DATA_PATH = '/content/drive/MyDrive/Startups_Recognize/NLP_Preprocessed_Dataset.csv'
df = pd.read_csv('NLP_Preprocessed_Dataset.csv')

In [ ]:
df.head()

,name,motto,description,topics,url,topic_category
0,Flux Plugins,"Speed up WordPress with AI-powered image, SEO, and accessibility tools","Flux Plugins provides a suite of WordPress plugins that boost speed, accessibility, and SEO. The Flux Suite includes Media Optimizer for compressing and serving images efficiently and AI Alt Text & Accessibility Audit to generate alt text and flag issues affecting compliance. Install in minutes, reduce bandwidth, and keep visitors engaged with faster, more accessible pages.","['Blogging Platforms', 'Performance Marketing', 'SEO']",https://betalist.com/startups/flux-plugins,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,Wafaa.io BOOSTED,Create digital contracts in minutes,"Wafaa is your new one-stop shop for digital contracts. Create, sign, and send contracts in minutes so you can get back to work. Replace scattered and unsecured agreements across platforms and manage the full contract lifecycle at a glance. Protect your money and time with built-in fraud prevention, and skip the subscriptions by paying as you go. Try before you buy: your first contract is on us.","['SaaS', 'B2B', 'Freelancers']",https://betalist.com/startups/wafaa-io,"[0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,Vaulternal,Store and automate file access with secure file storage,"Vaulternal offers encrypted, decentralized file storage with zero-knowledge privacy and smart triggers. Files are encrypted client-side and stored across multiple decentralized blockchains, including Polygon, removing single points of failure and preventing provider access. You can set time-based, inactivity, manual, or blockchain-event conditions to deliver files to chosen recipients, enabling controlled sharing and inheritance. Sign up with email or connect a wallet, manage multiple recipients, and rely on verifiable permanence and client-side decryption.","['Lifestyle', 'SaaS']",https://betalist.com/startups/vaulternal,"[0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,PUBVOICE,Turn your articles into audio and boost reader engagement,"PUBVOICE converts your articles into high-quality audio and delivers it through an embeddable player. Connect your RSS feed to auto-detect new posts, let AI generate listener-friendly scripts, and choose from 30 adjustable voices. Control creation, delivery, and performance from a single dashboard with metrics like plays, dwell time, and completion rate. Customize the player with CSS, clone voices, and keep content private from AI training. Implementation takes minutes with a lightweight script tag, and early access grants all features for free.",['Analytics'],https://betalist.com/startups/pubvoice,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,OPT-IMG,"AI tool to optimize images for SEO, speed, and scale automatically","OPT-IMG is an AI-powered image SEO platform that turns r

# Encoding Data

In [ ]:
print(df.topics[0])

['Blogging Platforms', 'Performance Marketing', 'SEO']


In [ ]:
type(df.topics[0])

str

In [ ]:
list_category = df.topics.to_list()
encode_count = {}

for topics in list_category:
  topics = topics.replace("[", "").replace("'", "").replace("]", "")
  topics_list = topics.split(",")
  for topic in topics_list:
    topic = topic.strip()
    if topic in encode_count.keys():
      encode_count[topic] += 1
    else:
      encode_count[topic] = 1

In [ ]:
#print(encode_count)

In [ ]:
encode_topic_types = {key: idx for idx, (key, value) in enumerate(encode_count.items())}

In [ ]:
#encode_topic_types

In [ ]:
labels = list(encode_topic_types.keys())
len(labels), labels[:5]

(111, ['Blogging Platforms', 'Performance Marketing', 'SEO', 'SaaS', 'B2B'])

# Splitting of Datasets

In [ ]:
splitter = RandomSplitter(valid_pct=0.1, seed=42)
train_ids, valid_ids = splitter(df)
len(train_ids), len(valid_ids)

(19218, 2135)

In [ ]:
valid_df = df.loc[valid_ids]
valid_df.head()

,name,motto,description,topics,url,topic_category
9287,10X,Build a community on your website,"10X enables anyone to add an engaging social interface to their website with one line of code. Returning more frequently to find fresh and interesting content promotes brand loyalty, peer to peer marketing, and on-site conversions.","['Brand Marketing', 'Web Tools', 'Social Media Marketing']",https://betalist.com/startups/10x,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1868,Bpzoo,Free professional graphics & online coloring platform,"Bpzoo is your free online graphics provider - powered by a basic color tool. We offer thousands of free and premium vector graphics on our platform, for both commercial and personal use.","['Design', 'Web Design', 'Graphics']",https://betalist.com/startups/bpzoo,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
14883,Internal,Monitor you and your team's well-being,Internal is a Slack bot that facilitates emotional communication for remote teams. All users take twice daily questionnaires to check in at the start of the day and reflect at the end of their day. Individual check in overall scores are reported to a public channel so that coworkers gather the same emotional insights that they would from body language in a physical setting. Reflections are used to track personal emotional fitness overtime.,"['SaaS', 'Business Productivity', 'Analytics']",https://betalist.com/startups/internal,"[0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
10067,fileGOD,Process PDFs and images in your browser without uploading,"fileGOD provides fast browser-based tools for PDFs and images that keep files on your device. You can compress, convert, merge, split, resize, crop, watermark, and more without uploading, protecting your privacy. It offers over 30 utilities including PDF to JPG, HEIC to JPG, OCR, and ZIP create/extract, supporting files up to 2GB. Start free with weekly actions and upgrade for heavier use.","['Document Management', 'Productivity Software', 'Business Productivity', 'Web Tools', 'SaaS']",https://betalist.com/startups/filegod,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
14941,Narrowr,Share one link per day - Build a feed you enjoy - Discover great content,"Narrowr is a social link-sharing platform that puts users first. We want people to think about what they share, opposed to mindlessly putting everything out there. Every member gets to share only one link per day – This constraint aims to promote quality over quantity and make feeds less addictive by nature. No algorithms, no ads, no tracking. It's all about our members and the content they enjoy.","['SaaS', 'Social Media']",https://betalist.com/startups/narrowr,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# Fastai & Blurr model load

In [ ]:
model_path = "models-roberta-base/book-roberta-base-classifier-stage-1.pkl"
learner_inf = load_learner(model_path)

In [ ]:
learner_inf.blurr_predict("random placeholder")

[{'labels': ['Web Tools', 'Web Development'],
  'scores': [0.8566012978553772, 0.8267977237701416],
  'class_indices': [0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0],
  'class_labels': ['Blogging Platforms', 'Performance Marketing', 'SEO', 'SaaS', 'B2B', 'Freelancers', 'Lifestyle', 'Analytics', 'Task Management', 'Productivity Software', 'Business Productivity', 'Application Perfo

In [ ]:
learner_inf.blurr_predict("random placeholder")[0]['labels']

['Web Tools', 'Web Development']

## Evaluation Metrics

In [ ]:
from sklearn import metrics

In [ ]:
def metric_measures(test_df, preds):

  targets = [np.asarray(target) for target in test_df['topic_category'].to_list()]
  outputs = [np.asarray(pred) for pred in preds]


  accuracy = metrics.accuracy_score(targets, outputs)
  f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
  f1_score_macro = metrics.f1_score(targets, outputs, average='macro')

  print(f"F1 Score (Micro) = {f1_score_micro}")
  print(f"F1 Score (Macro) = {f1_score_macro}")

  return

In [ ]:
preds = []
for idx, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
  desc = row['description']
  labels = learner_inf.blurr_predict(desc)[0]['class_indices']
  # pred_genres = [0] * len(encode_genre_types)
  # for label in labels:
  #   pred_genres[encode_genre_types[label]] = 1
  preds.append(labels)

preds[0][:20]

  0%|          | 0/2135 [00:00<?, ?it/s]

[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [ ]:
import ast

In [ ]:
# Create a copy to avoid modifying the original valid_df if it's used elsewhere
valid_df_processed = valid_df.copy()
# Convert the 'topic_category' column from string representation of lists to actual lists
valid_df_processed['topic_category'] = valid_df_processed['topic_category'].apply(ast.literal_eval)

metric_measures(valid_df_processed, preds)

F1 Score (Micro) = 0.7590176002141471
F1 Score (Macro) = 0.2934824248575889


# Convert fastai & blurr model to ONNX

In [ ]:
model_path = "models-roberta-base/book-roberta-base-classifier-stage-1.pkl"
learner_inf = load_learner(model_path)

In [ ]:
learner_inf.model.hf_model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [ ]:
classifier = learner_inf.model.hf_model.eval()

torch.onnx.export(
    classifier,
    torch.LongTensor([[0] * 512])
    'models/book-classifier.onnx',
    verbose=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['output'],
    opset_version=18,
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_len'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_len'},
        'output': {0: 'batch_size'}
    }
)

[torch.onnx] Obtain model graph for `RobertaForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `RobertaForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch_size,sequence_len]>,
                %"attention_mask"<INT64,[batch_size,sequence_len]>
            ),
            outputs=(
                %"output"<FLOAT,[1,111]>
            ),
            initializers=(
                %"roberta.embeddings.word_embeddings.weight"<FLOAT,[50265,768]>{TorchTensor(...)},
                %"roberta.embeddings.token_type_embeddings.weight"<FLOAT,[1,768]>{TorchTensor(...)},
                %"roberta.embeddings.LayerNorm.weight"<FLOAT,[768]>{TorchTensor(...)},
                %"roberta.embeddings.LayerNorm.bias"<FLOAT,[768]>{TorchTensor(...)},
                %"roberta.embeddings.position_embeddings.weight"<FL

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

In [ ]:
#onnx_model_path = 'models/book-classifier.onnx'
#quantized_onnx_model_path = 'models/book-classifier-quantized.onnx'

#quantize_dynamic(
#    onnx_model_path,
 #   quantized_onnx_model_path,
#    weight_type=QuantType.QUInt8,
#)

InferenceError: [ShapeInferenceError] Inferred shape and existing shape differ in dimension 0: (768) vs (111)

In [ ]:
dummy_input_ids = torch.LongTensor([[0] * 512])
dummy_attention_mask = torch.ones(1, 512, dtype=torch.long)

torch.onnx.export(
    classifier,
    (dummy_input_ids, dummy_attention_mask),   # tuple not single tensor
    'models/book-classifier.onnx',
    input_names=['input_ids', 'attention_mask'],
    output_names=['output'],
    opset_version=18,
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_len'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_len'},
        'output': {0: 'batch_size'}
    }
)

[torch.onnx] Obtain model graph for `RobertaForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `RobertaForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch_size,sequence_len]>,
                %"attention_mask"<INT64,[batch_size,sequence_len]>
            ),
            outputs=(
                %"output"<FLOAT,[1,111]>
            ),
            initializers=(
                %"roberta.embeddings.word_embeddings.weight"<FLOAT,[50265,768]>{TorchTensor(...)},
                %"roberta.embeddings.token_type_embeddings.weight"<FLOAT,[1,768]>{TorchTensor(...)},
                %"roberta.embeddings.LayerNorm.weight"<FLOAT,[768]>{TorchTensor(...)},
                %"roberta.embeddings.LayerNorm.bias"<FLOAT,[768]>{TorchTensor(...)},
                %"roberta.embeddings.position_embeddings.weight"<FL

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

In [ ]:
import onnx
#from onnxruntime.quantization import quantize_dynamic, QuantType

In [ ]:
!pip install -q onnxconverter-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.8 MB/s eta 0:00:00


In [ ]:
import onnx
from onnxconverter_common import float16

model = onnx.load('models/book-classifier.onnx')
model_fp16 = float16.convert_float_to_float16(model)
onnx.save(model_fp16, 'models/book-classifier-fp16.onnx')

# Testing of ONNX Inference

## Normal ONNX

In [ ]:
import onnxruntime as rt
from transformers import AutoTokenizer
import torch

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

class_labels = list(encode_topic_types.keys())

inf_session = rt.InferenceSession('models-roberta-base/book-roberta-base-classifier.onnx')
input_name = inf_session.get_inputs()[0].name
output_name = inf_session.get_outputs()[0].name

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
preds = []
for idx, row in tqdm(valid_df.iterrows(), total=valid_df.shape[0]):
  desc = row['description']
  input_ids = tokenizer(desc)['input_ids'][:512]

  probs = inf_session.run([output_name], {input_name: [input_ids]})[0]
  probs = torch.FloatTensor(probs)

  masks = torch.sigmoid(probs) >= 0.5
  labels = [class_labels[idx] for idx, mask in enumerate(masks[0]) if mask]

  pred_topics = [0] * len(encode_topic_types)
  for label in labels:
    pred_topics[encode_topic_types[label]] = 1
  preds.append(pred_topics)

  0%|          | 0/2135 [00:00<?, ?it/s]

In [ ]:
valid_df_processed = valid_df.copy()
valid_df_processed['topic_category'] = valid_df_processed['topic_category'].apply(ast.literal_eval)

metric_measures(valid_df_processed, preds)

F1 Score (Micro) = 0.7288135593220338
F1 Score (Macro) = 0.2692030142935526


## Quantized ONNX

In [ ]:
print(os.listdir('models'))

['book-distillBert-classifier-stage-0.pth', 'book-distillBert-classifier-stage-1.pth', 'book-MiniLM-L12-classifier-stage-0.pth', 'book-MiniLM-L12-classifier-stage-1.pth', 'book-roberta-base-classifier-stage-0.pth', 'book-roberta-base-classifier-stage-1.pth', 'book-classifier.onnx.data', 'book-classifier.onnx', 'book-classifier-fixed.onnx', 'book-classifier-fp16.onnx']


In [ ]:
#import os
print(os.listdir('models'))
print(os.listdir('models-roberta-base'))

['book-distillBert-classifier-stage-0.pth', 'book-distillBert-classifier-stage-1.pth', 'book-MiniLM-L12-classifier-stage-0.pth', 'book-MiniLM-L12-classifier-stage-1.pth', 'book-roberta-base-classifier-stage-0.pth', 'book-roberta-base-classifier-stage-1.pth', 'book-classifier.onnx.data', 'book-classifier.onnx', 'book-classifier-fixed.onnx', 'book-classifier-fp16.onnx']
['book-roberta-base-classifier-stage-0.pkl', 'book-roberta-base-classifier-stage-1.pkl', 'book-roberta-base-classifier-inferred.onnx', 'book-roberta-base-classifier.onnx.data', 'book-roberta-base-classifier.onnx']


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

class_labels = list(encode_topic_types.keys())

inf_session = rt.InferenceSession('models/book-classifier.onnx')
input_name = inf_session.get_inputs()[0].name
output_name = inf_session.get_outputs()[0].name

In [ ]:
preds = []
for idx, row in tqdm(valid_df.iterrows(), total=valid_df.shape[0]):
  desc = row['description']
  encoded = tokenizer(desc, truncation=True, max_length=512, return_tensors='np')
  probs = inf_session.run(
    [output_name],
    {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask']
    }
)[0]
  probs = torch.FloatTensor(probs)

  masks = torch.sigmoid(probs) >= 0.5
  labels = [class_labels[idx] for idx, mask in enumerate(masks[0]) if mask]

  pred_topics = [0] * len(encode_topic_types)
  for label in labels:
    pred_topics[encode_topic_types[label]] = 1
  preds.append(pred_topics)

  0%|          | 0/2135 [00:00<?, ?it/s]

In [ ]:
valid_df_processed = valid_df.copy()
valid_df_processed['topic_category'] = valid_df_processed['topic_category'].apply(ast.literal_eval)

metric_measures(valid_df_processed, preds)

F1 Score (Micro) = 0.7288135593220338
F1 Score (Macro) = 0.2692030142935526
